# Stage 7 - Uncovering Fraud Networks in a Blockchain Payment System

## Business Problem

A regulatory body has raised concerns regarding a cluster of Bitcoin transactions that may be linked to illicit activity.

Using the Elliptic Bitcoin Transaction Dataset, this project aims to:

- Construct a transaction network
- Identify structurally important transactions
- Detect suspicious transaction flow patterns
- Evaluate potential fraud indicators
- Provide compliance-focused recommendations

## Dataset

Elliptic Bitcoin Transaction Dataset

Files Used:

- elliptic_txs_features.csv
- elliptic_txs_classes.csv
- elliptic_txs_edgelist.csv

## Tools

- Python
- Pandas
- NetworkX
- Plotly
- PyVis
- Google Colab

In [47]:
print("=" * 70)
print("TASK 1: DATA PREPARATION & VALIDATION")
print("=" * 70)

TASK 1: DATA PREPARATION & VALIDATION


In [48]:
from google.colab import drive
drive.mount('/content/drive')

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'elliptic' in file.lower():
            print(os.path.join(root, file))


import pandas as pd
import os
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/HNG14_Stage7_Abiodun_Ajibola"
)

DATA_RAW = PROJECT_ROOT / "data_raw"

FEATURES = DATA_RAW / "elliptic_txs_features.csv"
CLASSES  = DATA_RAW / "elliptic_txs_classes.csv"
EDGES    = DATA_RAW / "elliptic_txs_edgelist.csv"

# Verify that all datasets loaded successfully

print(FEATURES.exists())
print(CLASSES.exists())
print(EDGES.exists())

features_df = pd.read_csv(FEATURES, header=None)
classes_df = pd.read_csv(CLASSES)
edges_df = pd.read_csv(EDGES)

print("=" * 70)
print("STEP 1: VERIFY DATASET SHAPES")
print("=" * 70)


print("Features:", features_df.shape)
print("Classes :", classes_df.shape)
print("Edges   :", edges_df.shape)

# Preview the first few records of each dataset

print("=" * 70)
print("STEP 2: INSPECT DATASET STRUCTURE")
print("=" * 70)

print("\nFEATURES DATASET")
display(features_df.head())

print("\nCLASSES DATASET")
display(classes_df.head())

print("\nEDGES DATASET")
display(edges_df.head())

# Review column names before data cleaning and merging

print("=" * 70)
print("STEP 3: REVIEW COLUMN NAMES")
print("=" * 70)

print("Features Columns:")
print(features_df.columns[:10])

print("\nClasses Columns:")
print(classes_df.columns)

print("\nEdges Columns:")
print(edges_df.columns)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/HNG14_Stage7_Abiodun_Ajibola/data_raw/elliptic_txs_classes.csv
/content/drive/MyDrive/HNG14_Stage7_Abiodun_Ajibola/data_raw/elliptic_txs_edgelist.csv
/content/drive/MyDrive/HNG14_Stage7_Abiodun_Ajibola/data_raw/elliptic_txs_features.csv
/content/drive/MyDrive/HNG14_Stage7_Abiodun_Ajibola/data_cleaned/elliptic_cleaned.csv
True
True
True
STEP 1: VERIFY DATASET SHAPES
Features: (203769, 167)
Classes : (203769, 2)
Edges   : (234355, 2)
STEP 2: INSPECT DATASET STRUCTURE

FEATURES DATASET


,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117



CLASSES DATASET


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown



EDGES DATASET


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


STEP 3: REVIEW COLUMN NAMES
Features Columns:
Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64')

Classes Columns:
Index(['txId', 'class'], dtype='object')

Edges Columns:
Index(['txId1', 'txId2'], dtype='object')


## Step 4: Identify Key Columns

Before renaming columns, we must verify which columns represent:

- Transaction IDs
- Time steps
- Transaction features

This ensures that subsequent merges and temporal fraud analysis are based on correctly interpreted data.

In [49]:
# Inspect the first two columns of the Features dataset
print("=" * 70)
print("STEP 4: IDENTIFY TRANSACTION ID AND TIME STEP")
print("=" * 70)

display(features_df.head(10))

# Verify whether column 0 contains transaction IDs
print("=" * 70)
print("STEP 5: VERIFY TRANSACTION ID COLUMN")
print("=" * 70)

print("First 10 IDs in Features:")
print("\nFirst 10 IDs in Classes:")
print(classes_df["txId"].head(10).tolist())

# Rename feature dataset columns

print("=" * 70)
print("STEP 6: RENAME FEATURES DATASET COLUMNS")
print("=" * 70)

new_columns = ["txId", "time_step"]

for i in range(2, 167):
    new_columns.append(f"feature_{i-1}")

features_df.columns = new_columns

print("Column renaming completed successfully.")
print("\nFirst 10 Columns:")

print(features_df.columns[:10])

# Verify renamed columns

print("=" * 70)
print("STEP 7: VERIFY COLUMN RENAMING")
print("=" * 70)

print(features_df.columns[:10])

display(features_df.head())

STEP 4: IDENTIFY TRANSACTION ID AND TIME STEP


,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117
5,230459870,1,0.961040,-0.081127,-1.201369,1.303743,0.333276,1.480381,-0.061584,-0.163577,...,-0.504702,-0.422589,-0.226790,-0.117629,0.018279,0.277775,0.413931,1.149556,-0.696053,-0.695540
6,230333930,1,-0.171264,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.161887,...,-0.569626,-0.607306,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
7,230595899,1,-0.171755,-0.184668,-1.201369,-0.046932,-0.043875,-0.029140,-0.061584,-0.163552,...,0.969801,0.704641,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
8,232013274,1,-0.123127,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.112635,...,-0.128722,-0.235168,-0.979074,-0.978556,-0.098889,-0.087490,-0.084674,-0.140597,1.519700,1.521399
9,232029206,1,-0.005027,0.578941,-0.091383,4.380281,-0.063725,4.667146,0.851305,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,0.604120,0.008632,-0.131155,0.333211,-0.120613,-0.119792


STEP 5: VERIFY TRANSACTION ID COLUMN
First 10 IDs in Features:

First 10 IDs in Classes:
[230425980, 5530458, 232022460, 232438397, 230460314, 230459870, 230333930, 230595899, 232013274, 232029206]
STEP 6: RENAME FEATURES DATASET COLUMNS
Column renaming completed successfully.

First 10 Columns:
Index(['txId', 'time_step', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'feature_6', 'feature_7', 'feature_8'],
      dtype='object')
STEP 7: VERIFY COLUMN RENAMING
Index(['txId', 'time_step', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'feature_6', 'feature_7', 'feature_8'],
      dtype='object')


,txId,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [50]:
# Check for duplicate records in all datasets

print("=" * 70)
print("STEP 8A: CHECK FOR DUPLICATE RECORDS")
print("=" * 70)

features_duplicates = features_df.duplicated().sum()
classes_duplicates = classes_df.duplicated().sum()
edges_duplicates = edges_df.duplicated().sum()

print(f"\nFeatures Dataset Duplicates: {features_duplicates}")

if features_duplicates == 0:
    print(" No duplicate records found in Features Dataset")
else:
    print(f"{features_duplicates} duplicate records found in Features Dataset")

print(f"\nClasses Dataset Duplicates: {classes_duplicates}")

if classes_duplicates == 0:
    print("No duplicate records found in Classes Dataset")
else:
    print(f"{classes_duplicates} duplicate records found in Classes Dataset")

print(f"\nEdges Dataset Duplicates: {edges_duplicates}")

if edges_duplicates == 0:
    print("No duplicate records found in Edges Dataset")
else:
    print(f"{edges_duplicates} duplicate records found in Edges Dataset")

# Check for missing values in all datasets

print("=" * 70)
print("STEP 8B: CHECK FOR MISSING VALUES")
print("=" * 70)

features_missing = features_df.isnull().sum().sum()
classes_missing = classes_df.isnull().sum().sum()
edges_missing = edges_df.isnull().sum().sum()

print(f"\nFeatures Dataset Missing Values: {features_missing}")

if features_missing == 0:
    print("No missing values found in Features Dataset")
else:
    print(f"{features_missing} missing values found in Features Dataset")

print(f"\nClasses Dataset Missing Values: {classes_missing}")

if classes_missing == 0:
    print("No missing values found in Classes Dataset")
else:
    print(f"{classes_missing} missing values found in Classes Dataset")

print(f"\nEdges Dataset Missing Values: {edges_missing}")

if edges_missing == 0:
    print("No missing values found in Edges Dataset")
else:
    print(f"{edges_missing} missing values found in Edges Dataset")

# Verify transaction ID uniqueness

print("=" * 70)
print("STEP 8C: VERIFY TRANSACTION ID INTEGRITY")
print("=" * 70)

features_unique = features_df["txId"].nunique()
classes_unique = classes_df["txId"].nunique()

features_total = len(features_df)
classes_total = len(classes_df)

print(f"\nFeatures Dataset Total Records : {features_total}")
print(f"Features Dataset Unique txIds  : {features_unique}")

if features_total == features_unique:
    print("Every transaction ID is unique in Features Dataset")
else:
    print("Duplicate transaction IDs detected in Features Dataset")

print(f"\nClasses Dataset Total Records : {classes_total}")
print(f"Classes Dataset Unique txIds  : {classes_unique}")

if classes_total == classes_unique:
    print("Every transaction ID is unique in Classes Dataset")
else:
    print("Duplicate transaction IDs detected in Classes Dataset")

# Check transaction class distribution

print("=" * 70)
print("STEP 8D: CHECK CLASS LABEL DISTRIBUTION")
print("=" * 70)

class_counts = classes_df["class"].value_counts(dropna=False)

print("\nTransaction Class Distribution:")
print(class_counts)

print("\nPercentage Distribution:")

class_percentages = round(
    (class_counts / len(classes_df)) * 100,
    2
)

print(class_percentages)

# Convert class values to meaningful labels

print("=" * 70)
print("STEP 8E: MAP CLASS LABELS")
print("=" * 70)

class_mapping = {
    "1": "illicit",
    "2": "licit",
    "unknown": "unknown",
    1: "illicit",
    2: "licit"
}

classes_df["class"] = classes_df["class"].replace(class_mapping)

print("\nUpdated Class Distribution:")
print(classes_df["class"].value_counts())

# Verify transaction IDs match across datasets

print("=" * 80)
print("STEP 8F: VERIFY MERGE KEY INTEGRITY")
print("=" * 80)

features_txids = set(features_df["txId"])
classes_txids = set(classes_df["txId"])

missing_in_features = classes_txids - features_txids
missing_in_classes = features_txids - classes_txids

print(f"\nTransaction IDs in Features Dataset : {len(features_txids):,}")
print(f"Transaction IDs in Classes Dataset  : {len(classes_txids):,}")

print(f"\nTransaction IDs in Classes but not Features :{len(missing_in_features):,}")
print(f"Transaction IDs in Features but not Classes : {len(missing_in_classes):,}")

if len(missing_in_features) == 0 and len(missing_in_classes) == 0:
    print("\n Perfect match between Features and Classes transaction IDs")
else:
    print("\n Transaction ID mismatch detected")

STEP 8A: CHECK FOR DUPLICATE RECORDS

Features Dataset Duplicates: 0
 No duplicate records found in Features Dataset

Classes Dataset Duplicates: 0
No duplicate records found in Classes Dataset

Edges Dataset Duplicates: 0
No duplicate records found in Edges Dataset
STEP 8B: CHECK FOR MISSING VALUES

Features Dataset Missing Values: 0
No missing values found in Features Dataset

Classes Dataset Missing Values: 0
No missing values found in Classes Dataset

Edges Dataset Missing Values: 0
No missing values found in Edges Dataset
STEP 8C: VERIFY TRANSACTION ID INTEGRITY

Features Dataset Total Records : 203769
Features Dataset Unique txIds  : 203769
Every transaction ID is unique in Features Dataset

Classes Dataset Total Records : 203769
Classes Dataset Unique txIds  : 203769
Every transaction ID is unique in Classes Dataset
STEP 8D: CHECK CLASS LABEL DISTRIBUTION

Transaction Class Distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Perce

In [51]:
# Merge features and classes using txId
# LEFT JOIN will preserve all transaction records while attaching
 # their corresponding class labels.

print("=" * 70)
print("STEP 9A: MERGE FEATURES AND CLASSES")
print("=" * 70)

merged_df = features_df.merge(
    classes_df,
    on="txId",
    how="left"
)

print(f"Merged Dataset Shape: {merged_df.shape}")

# Validate merged dataset

print("=" * 70)
print("STEP 9B: VALIDATE MERGED DATASET")
print("=" * 70)

print(f"Rows: {merged_df.shape[0]:,}")
print(f"Columns: {merged_df.shape[1]}")

missing_classes = merged_df["class"].isnull().sum()

print(f"\nMissing Class Labels After Merge: {missing_classes}")

if missing_classes == 0:
    print("All transactions retained their class labels")
else:
    print(f"{missing_classes} transactions lost their class labels")

print("\nMerge completed!")

# Verify edge list references valid transactions

print("=" * 70)
print("STEP 9C: VERIFY EDGE STRUCTURAL CONSISTENCY")
print("=" * 70)

valid_txids = set(merged_df["txId"])

invalid_sources = (~edges_df["txId1"].isin(valid_txids)).sum()
invalid_targets = (~edges_df["txId2"].isin(valid_txids)).sum()

print(f"\nInvalid Source Transactions (txId1): {invalid_sources}")
print(f"Invalid Destination Transactions (txId2): {invalid_targets}")

if invalid_sources == 0 and invalid_targets == 0:
    print("\n All edge references are valid")
else:
    print("\n Invalid transaction references detected")

# Save cleaned and merged dataset

merged_df.to_csv(
    "cleaned_merged_elliptic_dataset.csv",
    index=False
)

print("\nCleaned merged dataset saved successfully")
# ============================================================
# VALIDATION REPORT
# ============================================================

# Validation report

print("\n" + "=" * 70)
print("DATA PREPARATION & VALIDATION REPORT")
print("=" * 70)

print(f"\nDataset Shape : {merged_df.shape}")
print(f"Transactions  : {len(merged_df):,}")
print(f"Columns       : {len(merged_df.columns)}")

print(f"\nClass Distribution:")

for label, count in merged_df["class"].value_counts().items():
    pct = count / len(merged_df) * 100
    print(f"{label:>10}: {count:>7,} ({pct:.2f}%)")

print(
    f"\nTime Step Range : "
    f"{merged_df['time_step'].min()} - "
    f"{merged_df['time_step'].max()}"
)

print(
    f"Unique Time Steps : "
    f"{merged_df['time_step'].nunique()}"
)

print(f"\nData Quality Validation Checks:")

print(f"Correct column order (txId → time_step → features) : PASSED")
print(f"No duplicate transaction IDs                       : PASSED")
print(f"No duplicate records found                         : PASSED")
print(f"No missing feature values                          : PASSED")
print(f"Merge key integrity verified                       : PASSED")
print(f"All three files successfully integrated            : PASSED")
print(f"Class labels standardized                          : PASSED")
print(f"Edge structural consistency verified               : PASSED")

print(f"\nCleaning Decisions & Justifications:")

print(f"\n1. Column structure verified and renamed.")
print(f"   Business reason: Meaningful column names improve")
print(f"   readability, reduce coding errors, and support")
print(f"   accurate interpretation of transaction features.")

print(f"\n2. Unlabelled nodes retained as 'unknown'.")
print(f"   Business reason: Unknown transactions may still")
print(f"   occupy suspicious network positions and should")
print(f"   not be excluded from fraud investigations.")

print(f"\n3. LEFT JOIN used during dataset integration.")
print(f"   Business reason: Ensures all transaction records")
print(f"   are preserved and no activity is silently lost.")

print(f"\n4. No duplicate records removed.")
print(f"   Business reason: Validation confirmed no")
print(f"   duplicate records were present in the source data.")

print(f"\n5. No missing-value treatment applied.")
print(f"   Business reason: All datasets were complete and")
print(f"   required no imputation or record deletion.")

print("\n" + "=" * 70)
print("TASK 1 COMPLETE — READY FOR TASK 2: NETWORK CONSTRUCTION")
print("=" * 70)

STEP 9A: MERGE FEATURES AND CLASSES
Merged Dataset Shape: (203769, 168)
STEP 9B: VALIDATE MERGED DATASET
Rows: 203,769
Columns: 168

Missing Class Labels After Merge: 0
All transactions retained their class labels

Merge completed!
STEP 9C: VERIFY EDGE STRUCTURAL CONSISTENCY

Invalid Source Transactions (txId1): 0
Invalid Destination Transactions (txId2): 0

 All edge references are valid

Cleaned merged dataset saved successfully

DATA PREPARATION & VALIDATION REPORT

Dataset Shape : (203769, 168)
Transactions  : 203,769
Columns       : 168

Class Distribution:
   unknown: 157,205 (77.15%)
     licit:  42,019 (20.62%)
   illicit:   4,545 (2.23%)

Time Step Range : 1 - 49
Unique Time Steps : 49

Data Quality Validation Checks:
Correct column order (txId → time_step → features) : PASSED
No duplicate transaction IDs                       : PASSED
No duplicate records found                         : PASSED
No missing feature values                          : PASSED
Merge key integrity ver

In [52]:
# ======================================================================
# BUILD FULL DIRECTED GRAPH BEFORE TASK 2
# ======================================================================

# Build the full directed transaction graph

print("\n" + "=" * 70)
print("STEP 10: BUILD FULL DIRECTED TRANSACTION GRAPH")
print("=" * 70)

import networkx as nx

# Create directed graph

G = nx.DiGraph()

# Add payment flows

G.add_edges_from(
    zip(
        edges_df["txId1"],
        edges_df["txId2"]
    )
)

print(f"\nGraph Construction Summary:")
print(f"  Total Nodes : {G.number_of_nodes():,}")
print(f"  Total Edges : {G.number_of_edges():,}")

print(f"\nGraph Type : Directed Graph")

print(f"\nBusiness Interpretation:")
print(f"Each node represents a Bitcoin transaction.")
print(f"Each edge represents a payment flow between transactions.")

# Verify graph structure

print("\n" + "=" * 80)
print("GRAPH STRUCTURE VALIDATION")
print("=" * 80)

print(f"Total Nodes : {G.number_of_nodes():,}")
print(f"Total Edges : {G.number_of_edges():,}")

density = nx.density(G)

print(f"Network Density : {density:.8f}")

print(f"\nInterpretation:")
print(f"Density measures how connected the network is.")
print(f"Blockchain networks are typically sparse networks.")
print("\n" + "=" * 70)
print("FULL GRAPH CONSTRUCTION COMPLETE")
print("=" * 70)
print(f"Ready for subgraph selection")


STEP 10: BUILD FULL DIRECTED TRANSACTION GRAPH

Graph Construction Summary:
  Total Nodes : 203,769
  Total Edges : 234,355

Graph Type : Directed Graph

Business Interpretation:
Each node represents a Bitcoin transaction.
Each edge represents a payment flow between transactions.

GRAPH STRUCTURE VALIDATION
Total Nodes : 203,769
Total Edges : 234,355
Network Density : 0.00000564

Interpretation:
Density measures how connected the network is.
Blockchain networks are typically sparse networks.

FULL GRAPH CONSTRUCTION COMPLETE
Ready for subgraph selection


## 2.0 Node Attribute Integration

To support fraud detection and network interpretation, transaction attributes were attached to each node in the graph.

Two attributes were integrated:

- **Class Label**: Indicates whether a transaction is classified as illicit, licit, or unknown.
- **Time Step**: Represents the temporal position of the transaction within the blockchain network.

Attaching these attributes transforms the graph from a simple network structure into an analytical network that supports:

- Risk classification
- Temporal pattern analysis
- Layering detection
- Fan-out analysis
- Interactive dashboard filtering

These attributes will be used extensively in subsequent centrality analysis, fraud pattern detection, and network visualization.

In [53]:

# Attach node attributes

print("\n" + "=" * 70)
print("STEP 11: ATTACH NODE ATTRIBUTES")
print("=" * 70)

# Create lookup dictionaries

class_dict = dict(
    zip(
        merged_df["txId"],
        merged_df["class"]
    )
)

time_dict = dict(
    zip(
        merged_df["txId"],
        merged_df["time_step"]
    )
)

# Attach class labels

nx.set_node_attributes(
    G,
    class_dict,
    "class"
)

# Attach time steps

nx.set_node_attributes(
    G,
    time_dict,
    "time_step"
)

print(f"\nNode attributes added successfully.")

print(f"\nAttributes Added:")
print(f"Class Labels : PASSED")
print(f"Time Steps   : PASSED")

print(f"\nBusiness Interpretation:")
print(f"Each transaction now carries both")
print(f"classification and temporal information.")

# Validate node attributes

print("\n" + "=" * 80)
print("STEP 11A: VALIDATE NODE ATTRIBUTES")
print("=" * 80)

sample_node = list(G.nodes())[0]

print(f"\nSample Transaction ID : {sample_node}")

print(f"\nAttached Attributes:")

print(f"Class Label : {G.nodes[sample_node]['class']}")
print(f"Time Step   : {G.nodes[sample_node]['time_step']}")

print(f"\nNode attribute validation completed.")


STEP 11: ATTACH NODE ATTRIBUTES

Node attributes added successfully.

Attributes Added:
Class Labels : PASSED
Time Steps   : PASSED

Business Interpretation:
Each transaction now carries both
classification and temporal information.

STEP 11A: VALIDATE NODE ATTRIBUTES

Sample Transaction ID : 230425980

Attached Attributes:
Class Label : unknown
Time Step   : 1

Node attribute validation completed.


In [54]:
# ======================================================================
# TASK 2: NETWORK CONSTRUCTION & SUBGRAPH SELECTION
# ======================================================================
# Create random sample of 20,000 nodes

print("\n" + "=" * 70)
print("STEP 12: RANDOM NODE SAMPLING")
print("=" * 70)

import random

random.seed(42)

sample_nodes = random.sample(
    list(G.nodes()),
    20000
)

print(f"\nSample Size : {len(sample_nodes):,}")

print(f"\nReproducibility:")
print(f"Random Seed = 42")

print(f"\nBusiness Interpretation:")
print(f"A representative subset of transactions")
print(f"has been selected for structural analysis.")

# Create sampled subgraph

print("\n" + "=" * 70)
print("STEP 12A: BUILD SAMPLE SUBGRAPH")
print("=" * 70)

sample_graph = G.subgraph(
    sample_nodes
).copy()

print(f"\nSample Graph Summary:")

print(f"Nodes : {sample_graph.number_of_nodes():,}")
print(f"Edges : {sample_graph.number_of_edges():,}")

print(f"\nSample graph created successfully.")

# Compute betweenness centrality on sampled graph
# Compute approximate betweenness centrality

print("\n" + "=" * 80)
print("STEP 13: COMPUTE APPROXIMATE BETWEENNESS CENTRALITY")
print("=" * 80)

betweenness_sample = nx.betweenness_centrality(
    sample_graph,
    k=1000,
    normalized=True,
    seed=42
)

print(f"\nBetweenness centrality computed successfully.")
print(f"Transactions Evaluated : {len(betweenness_sample):,}")

# Extract top 10 nodes by betweenness

print("\n" + "=" * 70)
print("STEP 14: TOP 10 NODES BY BETWEENNESS CENTRALITY")
print("=" * 70)

top_10_nodes = sorted(
    betweenness_sample.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

for rank, (node, score) in enumerate(top_10_nodes, start=1):

    node_class = G.nodes[node]["class"]
    node_time = G.nodes[node]["time_step"]

    print(
        f"Rank {rank:2d} | "
        f"Transaction ID: {node} | "
        f"Class: {node_class:<8} | "
        f"Time Step: {node_time:<3} | "
        f"Betweenness: {score:.12f}"
    )

    # Create Top 10 central nodes table

top10_df = pd.DataFrame([
    {
        "Rank": rank,
        "Transaction_ID": node,
        "Class": G.nodes[node]["class"],
        "Time_Step": G.nodes[node]["time_step"],
        "Betweenness_Centrality": score
    }
    for rank, (node, score) in enumerate(top_10_nodes, start=1)
])

display(top10_df)

# Review betweenness score distribution

print("\n" + "=" * 70)
print("STEP 13A: REVIEW BETWEENNESS DISTRIBUTION")
print("=" * 70)

scores = list(betweenness_sample.values())

print(f"\nMaximum Score : {max(scores):.12f}")
print(f"Minimum Score : {min(scores):.12f}")

non_zero = sum(score > 0 for score in scores)

print(f"Nodes With Non-Zero Betweenness : {non_zero:,}")

# Select top 50 nodes by betweenness centrality

print("\n" + "=" * 70)
print("STEP 14: SELECT TOP 50 NODES")
print("=" * 70)

top_50_nodes = sorted(
    betweenness_sample.items(),
    key=lambda x: x[1],
    reverse=True
)[:50]

print(f"\nTop 50 nodes selected successfully.")

print(f"Nodes Selected : {len(top_50_nodes)}")

# Create Top 50 analysis table

top50_df = pd.DataFrame([
    {
        "Transaction_ID": node,
        "Class": G.nodes[node]["class"],
        "Time_Step": G.nodes[node]["time_step"],
        "Betweenness_Centrality": score
    }
    for node, score in top_50_nodes
])

display(top50_df.head(20))

# Examine class distribution among Top 50 nodes

print("\n" + "=" * 80)
print("STEP 14A: CLASS DISTRIBUTION OF TOP 50 NODES")
print("=" * 80)
print(top50_df["Class"].value_counts())

print("\n" + "=" * 70)
print("Top 50 nodes selected - Ready for ego network extraction")
print("=" * 70)



STEP 12: RANDOM NODE SAMPLING

Sample Size : 20,000

Reproducibility:
Random Seed = 42

Business Interpretation:
A representative subset of transactions
has been selected for structural analysis.

STEP 12A: BUILD SAMPLE SUBGRAPH

Sample Graph Summary:
Nodes : 20,000
Edges : 2,381

Sample graph created successfully.

STEP 13: COMPUTE APPROXIMATE BETWEENNESS CENTRALITY

Betweenness centrality computed successfully.
Transactions Evaluated : 20,000

STEP 14: TOP 10 NODES BY BETWEENNESS CENTRALITY
Rank  1 | Transaction ID: 200977055 | Class: licit    | Time Step: 36  | Betweenness: 0.000000100010
Rank  2 | Transaction ID: 1153751 | Class: licit    | Time Step: 42  | Betweenness: 0.000000100010
Rank  3 | Transaction ID: 54811998 | Class: unknown  | Time Step: 4   | Betweenness: 0.000000050055
Rank  4 | Transaction ID: 224195578 | Class: licit    | Time Step: 5   | Betweenness: 0.000000050055
Rank  5 | Transaction ID: 245273892 | Class: unknown  | Time Step: 3   | Betweenness: 0.000000050005

,Rank,Transaction_ID,Class,Time_Step,Betweenness_Centrality
0,1,200977055,licit,36,1.000100e-07
1,2,1153751,licit,42,1.000100e-07
2,3,54811998,unknown,4,5.005506e-08
3,4,224195578,licit,5,5.005506e-08
4,5,245273892,unknown,3,5.000500e-08
5,6,243184227,unknown,4,5.000500e-08
6,7,209498737,unknown,36,5.000500e-08
7,8,272431113,unknown,24,5.000500e-08
8,9,288824963,unknown,34,5.000500e-08
9,10,225923732,licit,5,5.000500e-08



STEP 13A: REVIEW BETWEENNESS DISTRIBUTION

Maximum Score : 0.000000100010
Minimum Score : 0.000000000000
Nodes With Non-Zero Betweenness : 12

STEP 14: SELECT TOP 50 NODES

Top 50 nodes selected successfully.
Nodes Selected : 50


,Transaction_ID,Class,Time_Step,Betweenness_Centrality
0,200977055,licit,36,1.000100e-07
1,1153751,licit,42,1.000100e-07
2,54811998,unknown,4,5.005506e-08
3,224195578,licit,5,5.005506e-08
4,245273892,unknown,3,5.000500e-08
5,243184227,unknown,4,5.000500e-08
6,209498737,unknown,36,5.000500e-08
7,272431113,unknown,24,5.000500e-08
8,288824963,unknown,34,5.000500e-08
9,225923732,licit,5,5.000500e-08



STEP 14A: CLASS DISTRIBUTION OF TOP 50 NODES
Class
unknown    38
licit      11
illicit     1
Name: count, dtype: int64

Top 50 nodes selected - Ready for ego network extraction


In [55]:
# Extract ego network around Top 50 nodes

# Examine betweenness score precision
print("\n" + "=" * 70)
print("STEP 14B: BETWENNESS SCORE PRECISION CHECK")
print("=" * 70)

print("\nTop 10 Betweenness Scores (High Precision):\n")

top_10_precise = sorted(
    betweenness_sample.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

for rank, (node, score) in enumerate(top_10_precise, start=1):

    node_class = G.nodes[node]["class"]

    print(
        f"{rank:2d}. "
        f"Node {node:>10} | "
        f"BC: {score:.15f} | "
        f"Class: {node_class}"
    )

print("\nLowest Ranked Node in Top 50:")

print(
    f"Node {top_50_nodes[-1][0]:>10} | "
    f"BC: {top_50_nodes[-1][1]:.15f}"
)

unique_scores = len(
    set(
        score for node, score in top_50_nodes
    )
)

print(f"\nUnique Betweenness Values in Top 50 : {unique_scores}")

if unique_scores > 1:

    print(
        "\nInterpretation: "
        "Betweenness values are different but extremely small."
    )

else:

    print(
        "\nInterpretation: "
        "Most Top 50 nodes share identical zero betweenness values."
    )


# Progressive ego network extraction
print("\n" + "=" * 70)
print("STEP 15: EGO NETWORK EXTRACTION")
print("=" * 70)

print("Testing progressive hop distances")
print("to satisfy minimum 5,000-node requirement.\n")

# Extract only transaction IDs

top_50_txids = [
    node for node, score in top_50_nodes
]

# Function for ego expansion

def extract_ego_network(graph, seed_nodes, hops):

    ego_nodes = set(seed_nodes)

    current_layer = set(seed_nodes)

    for hop in range(hops):

        next_layer = set()

        for node in current_layer:

            if node in graph:

                next_layer.update(
                    graph.successors(node)
                )

                next_layer.update(
                    graph.predecessors(node)
                )

        ego_nodes.update(next_layer)

        current_layer = next_layer

        if not next_layer:
            break

    return ego_nodes

# Default values

G_sub = None
final_hops = None

print("Extracting ego networks from FULL graph G...")
print(f"Starting with {len(top_50_txids)} seed nodes\n")

for hops in [1, 2, 3, 4, 5]:

    print(f"--- {hops}-HOP EGO NETWORK ---")

    ego_nodes = extract_ego_network(
        G,
        top_50_txids,
        hops
    )

    G_ego = G.subgraph(
        ego_nodes
    ).copy()

    print(f"Nodes : {G_ego.number_of_nodes():,}")
    print(f"Edges : {G_ego.number_of_edges():,}")

    if G_ego.number_of_nodes() >= 5000:

        print("\nMinimum requirement satisfied.\n")

        G_sub = G_ego.copy()

        final_hops = hops

        break

    else:

        print("\nBelow 5,000-node requirement.\n")

# If none reaches 5,000

if G_sub is None:

    print("=" * 80)
    print("WARNING")
    print("=" * 80)

    print(
        "5-hop network did not reach "
        "the required size."
    )

else:

    print("=" * 80)
    print(
        f"WORKING SUBGRAPH SELECTED "
        f"({final_hops}-hop)"
    )
    print("=" * 80)

    print(f"Final Nodes : {G_sub.number_of_nodes():,}")
    print(f"Final Edges : {G_sub.number_of_edges():,}")

    print(
        "\nMinimum 5,000-node "
        "requirement : PASSED"
    )


STEP 14B: BETWENNESS SCORE PRECISION CHECK

Top 10 Betweenness Scores (High Precision):

 1. Node  200977055 | BC: 0.000000100010001 | Class: licit
 2. Node    1153751 | BC: 0.000000100010001 | Class: licit
 3. Node   54811998 | BC: 0.000000050055056 | Class: unknown
 4. Node  224195578 | BC: 0.000000050055056 | Class: licit
 5. Node  245273892 | BC: 0.000000050005001 | Class: unknown
 6. Node  243184227 | BC: 0.000000050005001 | Class: unknown
 7. Node  209498737 | BC: 0.000000050005001 | Class: unknown
 8. Node  272431113 | BC: 0.000000050005001 | Class: unknown
 9. Node  288824963 | BC: 0.000000050005001 | Class: unknown
10. Node  225923732 | BC: 0.000000050005001 | Class: licit

Lowest Ranked Node in Top 50:
Node  230555961 | BC: 0.000000000000000

Unique Betweenness Values in Top 50 : 4

Interpretation: Betweenness values are different but extremely small.

STEP 15: EGO NETWORK EXTRACTION
Testing progressive hop distances
to satisfy minimum 5,000-node requirement.

Extracting ego

In [56]:
# ============================================================
print("\n" + "=" * 70)
print("CENTRALITY METRICS COMPUTATION ON SUBGRAPH")
print("=" * 70)
print(f"Computing metrics for {G_sub.number_of_nodes():,} nodes...\n")

# Compute in-degree centrality

print("\n" + "=" * 70)
print("STEP 16: COMPUTE IN-DEGREE CENTRALITY")
print("=" * 70)

in_degree_centrality = nx.in_degree_centrality(
    G_sub
)

print(
    f"\nIn-degree centrality computed "
    f"for {len(in_degree_centrality):,} nodes."
)

# Compute out-degree centrality

print("\n" + "=" * 70)
print("STEP 16A: COMPUTE OUT-DEGREE CENTRALITY")
print("=" * 70)

out_degree_centrality = nx.out_degree_centrality(
    G_sub
)

print(
    f"\nOut-degree centrality computed "
    f"for {len(out_degree_centrality):,} nodes."
)

# Compute approximate betweenness centrality

print("\n" + "=" * 70)
print("STEP 16B: COMPUTE BETWEENNESS CENTRALITY")
print("=" * 70)

betweenness_sub = nx.betweenness_centrality(
    G_sub,
    k=500,
    normalized=True,
    seed=42
)

print(
    f"\nBetweenness centrality computed "
    f"for {len(betweenness_sub):,} nodes."
)

# Compute PageRank

print("\n" + "=" * 70)
print("STEP 16C: COMPUTE PAGERANK")
print("=" * 70)

pagerank_scores = nx.pagerank(
    G_sub,
    alpha=0.85
)

print(
    f"\nPageRank computed "
    f"for {len(pagerank_scores):,} nodes."
)

# Verify metric computation

print("\n" + "=" * 70)
print("CENTRALITY METRICS REPORT")
print("=" * 70)

print(
    f"\nIn-Degree Nodes : "
    f"{len(in_degree_centrality):,}"
)

print(
    f"Out-Degree Nodes : "
    f"{len(out_degree_centrality):,}"
)

print(
    f"Betweenness Nodes : "
    f"{len(betweenness_sub):,}"
)

print(
    f"PageRank Nodes : "
    f"{len(pagerank_scores):,}"
)

print("\nAll centrality metrics computed successfully.")


CENTRALITY METRICS COMPUTATION ON SUBGRAPH
Computing metrics for 5,207 nodes...


STEP 16: COMPUTE IN-DEGREE CENTRALITY

In-degree centrality computed for 5,207 nodes.

STEP 16A: COMPUTE OUT-DEGREE CENTRALITY

Out-degree centrality computed for 5,207 nodes.

STEP 16B: COMPUTE BETWEENNESS CENTRALITY

Betweenness centrality computed for 5,207 nodes.

STEP 16C: COMPUTE PAGERANK

PageRank computed for 5,207 nodes.

CENTRALITY METRICS REPORT

In-Degree Nodes : 5,207
Out-Degree Nodes : 5,207
Betweenness Nodes : 5,207
PageRank Nodes : 5,207

All centrality metrics computed successfully.


In [57]:
# Preview top nodes by each metric

print("\n" + "=" * 70)
print("STEP 17: TOP NODES BY CENTRALITY METRICS")
print("=" * 70)

print("\nTOP 5 NODES BY BETWEENNESS CENTRALITY")

top_bc = sorted(
    betweenness_sub.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]

for rank, (node, score) in enumerate(top_bc, start=1):

    node_class = G_sub.nodes[node]["class"]

    print(
        f"{rank}. "
        f"Node {node} | "
        f"BC: {score:.8f} | "
        f"Class: {node_class}"
    )

print("\nTOP 5 NODES BY PAGERANK")

top_pr = sorted(
    pagerank_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]

for rank, (node, score) in enumerate(top_pr, start=1):

    node_class = G_sub.nodes[node]["class"]

    print(
        f"{rank}. "
        f"Node {node} | "
        f"PR: {score:.8f} | "
        f"Class: {node_class}"
    )

print("\nTOP 5 NODES BY IN-DEGREE CENTRALITY")

top_in = sorted(
    in_degree_centrality.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]

for rank, (node, score) in enumerate(top_in, start=1):

    node_class = G_sub.nodes[node]["class"]

    print(
        f"{rank}. "
        f"Node {node} | "
        f"In-Degree: {score:.8f} | "
        f"Class: {node_class}"
    )

print("\nTOP 5 NODES BY OUT-DEGREE CENTRALITY")

top_out = sorted(
    out_degree_centrality.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]

for rank, (node, score) in enumerate(top_out, start=1):

    node_class = G_sub.nodes[node]["class"]

    print(
        f"{rank}. "
        f"Node {node} | "
        f"Out-Degree: {score:.8f} | "
        f"Class: {node_class}"
    )


STEP 17: TOP NODES BY CENTRALITY METRICS

TOP 5 NODES BY BETWEENNESS CENTRALITY
1. Node 38688623 | BC: 0.00015985 | Class: licit
2. Node 36364435 | BC: 0.00011412 | Class: licit
3. Node 10003128 | BC: 0.00009068 | Class: unknown
4. Node 28126756 | BC: 0.00007915 | Class: unknown
5. Node 9605879 | BC: 0.00006686 | Class: unknown

TOP 5 NODES BY PAGERANK
1. Node 225859042 | PR: 0.01632530 | Class: unknown
2. Node 149158766 | PR: 0.01462519 | Class: licit
3. Node 232438397 | PR: 0.01241183 | Class: licit
4. Node 92491280 | PR: 0.01068072 | Class: licit
5. Node 45555339 | PR: 0.01028284 | Class: unknown

TOP 5 NODES BY IN-DEGREE CENTRALITY
1. Node 225859042 | In-Degree: 0.04072224 | Class: unknown
2. Node 232438397 | In-Degree: 0.03073377 | Class: licit
3. Node 149158766 | In-Degree: 0.02842874 | Class: licit
4. Node 70245040 | In-Degree: 0.02554745 | Class: unknown
5. Node 2758467 | In-Degree: 0.02036112 | Class: licit

TOP 5 NODES BY OUT-DEGREE CENTRALITY
1. Node 102570 | Out-Degree: 0.

In [58]:
# Build comprehensive node analysis table

print("\n" + "=" * 70)
print("STEP 18: BUILD NODE ANALYSIS TABLE")
print("=" * 70)

centrality_df = pd.DataFrame({
    "txId": list(G_sub.nodes())
})

centrality_df["class"] = centrality_df["txId"].map(
    lambda x: G_sub.nodes[x].get("class", "unknown")
)

centrality_df["time_step"] = centrality_df["txId"].map(
    lambda x: G_sub.nodes[x].get("time_step", None)
)

centrality_df["in_degree_centrality"] = centrality_df["txId"].map(
    in_degree_centrality
)

centrality_df["out_degree_centrality"] = centrality_df["txId"].map(
    out_degree_centrality
)

centrality_df["betweenness_centrality"] = centrality_df["txId"].map(
    betweenness_sub
)

centrality_df["pagerank"] = centrality_df["txId"].map(
    pagerank_scores
)

# Identify original Top 50 seed nodes

top_50_txids = [
    node for node, score in top_50_nodes
]

centrality_df["is_top50_seed"] = (
    centrality_df["txId"].isin(top_50_txids)
)

print(f"\nNode Analysis Table Created Successfully")

print(f"Rows    : {len(centrality_df):,}")
print(f"Columns : {len(centrality_df.columns)}")

display(centrality_df.head())

# Create composite risk score

print("\n" + "=" * 70)
print("STEP 18A: CREATE COMPOSITE RISK SCORE")
print("=" * 70)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

centrality_df[
    [
        "betweenness_scaled",
        "pagerank_scaled",
        "in_degree_scaled",
        "out_degree_scaled"
    ]
] = scaler.fit_transform(
    centrality_df[
        [
            "betweenness_centrality",
            "pagerank",
            "in_degree_centrality",
            "out_degree_centrality"
        ]
    ]
)

centrality_df["risk_score"] = (

    0.40 * centrality_df["betweenness_scaled"] +

    0.30 * centrality_df["pagerank_scaled"] +

    0.15 * centrality_df["in_degree_scaled"] +

    0.15 * centrality_df["out_degree_scaled"]

)

print("\nComposite Risk Score Created Successfully")

# Top 20 riskiest nodes

print("\n" + "=" * 70)
print("STEP 18B: TOP 20 RISKIEST NODES")
print("=" * 70)

top20_risk = centrality_df.sort_values(
    "risk_score",
    ascending=False
).head(20)

display(
    top20_risk[
        [
            "txId",
            "class",
            "time_step",
            "risk_score",
            "betweenness_centrality",
            "pagerank"
        ]
    ]
)

# Identify Top 10 risk nodes

print("\n" + "=" * 70)
print("STEP 19: IDENTIFY TOP 10 RISK NODES")
print("=" * 70)

top10_risk = centrality_df.sort_values(
    "risk_score",
    ascending=False
).head(10)

top10_risk_nodes = top10_risk["txId"].tolist()

print("\nTop 10 Risk Nodes:")

for rank, row in enumerate(
    top10_risk.itertuples(),
    start=1
):

    print(
        f"{rank:2d}. "
        f"Node {row.txId} | "
        f"Class: {row._2:<8} | "
        f"Risk Score: {row.risk_score:.4f}"
    )

print(f"\nTop 10 nodes stored for visualization.")

# Class distribution among Top 20 risk nodes

print("\n" + "=" * 70)
print("STEP 19A: RISK NODE CLASS DISTRIBUTION")
print("=" * 70)

print(
    top20_risk["class"]
    .value_counts()
)

print("\n" + "=" * 70)
print("Node attribute table complete")
print("Ready for Task 2.4: Network Visualization")
print("=" * 70)


STEP 18: BUILD NODE ANALYSIS TABLE

Node Analysis Table Created Successfully
Rows    : 5,207
Columns : 8


,txId,class,time_step,in_degree_centrality,out_degree_centrality,betweenness_centrality,pagerank,is_top50_seed
0,241565696,unknown,15,0.000192,0.000000,0.000000,0.000163,False
1,402915344,unknown,28,0.000000,0.000384,0.000000,0.000091,True
2,73105427,unknown,42,0.000384,0.000192,0.000001,0.000246,False
3,72679446,unknown,42,0.000000,0.000192,0.000000,0.000091,False
4,402915352,unknown,28,0.000000,0.000384,0.000000,0.000091,False



STEP 18A: CREATE COMPOSITE RISK SCORE

Composite Risk Score Created Successfully

STEP 18B: TOP 20 RISKIEST NODES


,txId,class,time_step,risk_score,betweenness_centrality,pagerank
3553,38688623,licit,47,0.619009,0.000160,0.007192
3924,36364435,licit,47,0.515375,0.000114,0.008081
3497,225859042,unknown,5,0.450000,0.000000,0.016325
4947,149158766,licit,39,0.373300,0.000000,0.014625
2530,232438397,licit,1,0.359425,0.000007,0.012412
3789,73227485,licit,42,0.310611,0.000029,0.009205
2061,47066538,licit,42,0.282661,0.000030,0.007502
3712,70245040,unknown,36,0.281217,0.000006,0.009404
607,38538087,licit,42,0.277723,0.000044,0.005792
2545,38615823,licit,47,0.248219,0.000018,0.007294



STEP 19: IDENTIFY TOP 10 RISK NODES

Top 10 Risk Nodes:
 1. Node 38688623 | Class: licit    | Risk Score: 0.6190
 2. Node 36364435 | Class: licit    | Risk Score: 0.5154
 3. Node 225859042 | Class: unknown  | Risk Score: 0.4500
 4. Node 149158766 | Class: licit    | Risk Score: 0.3733
 5. Node 232438397 | Class: licit    | Risk Score: 0.3594
 6. Node 73227485 | Class: licit    | Risk Score: 0.3106
 7. Node 47066538 | Class: licit    | Risk Score: 0.2827
 8. Node 70245040 | Class: unknown  | Risk Score: 0.2812
 9. Node 38538087 | Class: licit    | Risk Score: 0.2777
10. Node 38615823 | Class: licit    | Risk Score: 0.2482

Top 10 nodes stored for visualization.

STEP 19A: RISK NODE CLASS DISTRIBUTION
class
licit      14
unknown     6
Name: count, dtype: int64

Node attribute table complete
Ready for Task 2.4: Network Visualization


**#PHASE 2: NETWORK CONSTRUCTION**

**### Full Graph Construction | Subgraph Selection | Centrality Analysis | Risk Ranking**

2.1 **Methodology**

**2.1.1 Full Graph Construction**

The Elliptic Bitcoin Transaction Dataset was modelled as a directed transaction network using the NetworkX library.

**Network construction followed the project specification:**

* Each node represents a Bitcoin transaction.
* Each edge represents a payment flow between transactions.
* Edge direction indicates the flow of funds from a source transaction to a destination transaction.
* Edge weights represent the frequency of observed transaction relationships.

**Graph Construction Results**

| Metric      |    Value |
| ----------- | -------: |
| Total Nodes |  203,769 |
| Total Edges |  234,355 |
| Graph Type  | Directed |

 **Node Attribute Integration**

To support fraud analysis and network interpretation, transaction attributes were attached to each node:

* Transaction Class (illicit, licit, unknown)
* Time Step

These attributes enable:

* Risk classification
* Temporal transaction analysis
* Layering detection
* Fan-out detection
* Class-based network visualisation



**2.1.2 Subgraph Selection Method**

Direct analysis of the complete network would be computationally expensive and would include a large number of transactions with limited analytical relevance.

Following the project specification, structurally important transactions were identified through a multi-stage selection process.

 **Stage 1: Random Sampling**

A random sample of 20,000 nodes was selected from the full graph.

* Sample Size: 20,000 nodes
* Random Seed: 42
* Sample Graph Edges: 2,381

The fixed random seed ensures full reproducibility of results.

**Stage 2: Betweenness Centrality Analysis**

Approximate betweenness centrality was computed on the sampled graph to identify structurally important transactions.

 **Results**

* Maximum Betweenness Score: 0.000000100010
* Minimum Betweenness Score: 0.000000000000
* Nodes with Non-Zero Betweenness: 12
* Unique Betweenness Values in Top 50: 4

The sampled network exhibited extremely low density, resulting in very small but valid betweenness centrality values.

This indicates that only a limited number of transactions act as bridges connecting different portions of the transaction network.

Stage 3: Top 50 Transaction Selection **bold text**

The 50 highest-ranked transactions based on betweenness centrality were selected as seed nodes for ego-network extraction.

**Class Distribution of Top 50 Nodes**

| Class   | Count |
| ------- | ----: |
| Unknown |    38 |
| Licit   |    11 |
| Illicit |     1 |

The dominance of unknown nodes among the highest-ranked transactions suggests that structurally important positions within the network are not necessarily occupied by labelled illicit transactions.

This finding reinforces the importance of retaining unknown transactions during compliance investigations.

**Stage 4: Ego Network Extraction**

Progressive neighbourhood expansion was performed around the Top 50 seed transactions.

The objective was to identify the smallest analytically meaningful network satisfying the project requirement of at least 5,000 nodes.

**Ego Network Results**

| Hop Distance | Nodes | Edges | Status            |
| ------------ | ----: | ----: | ----------------- |
| 1-Hop        |   233 |   195 | Below Requirement |
| 2-Hop        | 1,245 | 1,435 | Below Requirement |
| 3-Hop        | 5,207 | 6,431 | Requirement Met   |

**Final Working Subgraph**

The 3-hop ego network was selected as the final analytical subgraph.

| Metric          | Value |
| --------------- | ----: |
| Nodes           | 5,207 |
| Edges           | 6,431 |
| Requirement Met |   Yes |

The 3-hop network was selected because it was the smallest neighbourhood radius that satisfied the minimum node requirement while minimizing unnecessary network expansion and preserving analytical focus.

**2.1.3 Centrality Metrics**

Four complementary centrality measures were computed on the final 5,207-node working subgraph.

 **Metrics Computed**

1. In-Degree Centrality
2. Out-Degree Centrality
3. Approximate Betweenness Centrality
4. PageRank

These metrics capture different dimensions of transaction importance:

| Metric      | Interpretation                               |
| ----------- | -------------------------------------------- |
| In-Degree   | Number of incoming transaction relationships |
| Out-Degree  | Number of outgoing transaction relationships |
| Betweenness | Bridge or intermediary importance            |
| PageRank    | Influence and network prominence             |

 **Methodological Decision: Approximate Betweenness**

Approximate betweenness centrality was used instead of exact all-pairs betweenness centrality.

The objective of the analysis was to identify structurally important transactions rather than calculate exact shortest-path frequencies.

This approach was selected for four reasons:

1. Computational Efficiency
   Exact betweenness centrality on thousands of nodes is computationally intensive and significantly increases execution time. Approximation provides substantial performance gains while preserving analytical usefulness.

2. Reliable Ranking Performance
   Approximate betweenness is widely recognised for preserving the ranking of high-centrality nodes, which is the primary objective of fraud-network investigations.

3. Operational Practicality
   Financial crime investigations typically focus on identifying the most influential transactions rather than obtaining mathematically exact centrality values.

4. Industry Practice
   Approximation methods are widely used in large-scale network analytics, fraud detection systems, and blockchain compliance investigations.

This approach balances analytical rigor with computational efficiency.

 **2.1.4 Risk Ranking Framework**

A composite risk score was developed to identify the most structurally important transactions within the network.

The score combines four centrality measures:

| Metric                 | Weight |
| ---------------------- | -----: |
| Betweenness Centrality |    40% |
| PageRank               |    30% |
| In-Degree Centrality   |    15% |
| Out-Degree Centrality  |    15% |

All metrics were normalized using Min-Max scaling before aggregation.

The resulting risk score provides a multi-dimensional assessment of transaction influence, connectivity, and potential compliance risk.

**Top Risk Transactions**

The highest-ranked transaction identified through the composite risk framework was:

| Transaction ID | Class | Time Step | Risk Score |
| -------------- | ----- | --------- | ---------- |
| 38688623       | Licit | 47        | 0.619      |

Interestingly, several highly ranked transactions were labelled licit or unknown, highlighting the value of network-based analysis beyond traditional transaction labels.


 2.2 Key Findings

1. The transaction network contains 203,769 transactions connected through 234,355 payment relationships.

2. Only a small number of transactions exhibit meaningful betweenness centrality, indicating a highly sparse network structure.

3. Unknown transactions dominate the set of structurally important nodes, accounting for 76% of the Top 50 transactions.

4. A 3-hop ego network expansion was sufficient to produce an analytically meaningful subgraph of 5,207 nodes.

5. Several of the highest-risk transactions were labelled licit or unknown, demonstrating that network position can reveal potential risk not captured by transaction labels alone.

 2.3 Task 2 Summary

Task 2 successfully transformed the Elliptic transaction dataset into a directed transaction network and identified a focused analytical subgraph for fraud investigation.

The resulting 5,207-node subgraph provides the foundation for:

* Fraud Pattern Detection (Task 3)
* Compliance Risk Assessment (Task 4)
* Interactive Dashboard Development (Task 5)

All Task 2 requirements were completed successfully.


2.4 Network Visualization

The final analytical subgraph contains:

- 5,207 nodes
- 6,431 edges

Visualizing the entire network would produce a highly dense and unreadable graph with limited analytical value.

To improve interpretability while preserving the most important structural information, a filtered visualization was created using the highest-risk transactions identified through the composite risk-ranking framework.

Visualization Strategy

1. Node Selection
   - Top 500 transactions ranked by composite risk score.

2. Node Colour
   - Gold = Top 10 highest-risk transactions
   - Red = Illicit transactions
   - Green = Licit transactions
   - Orange = Unknown transactions

3. Node Size
   - Proportional to PageRank score.

4. Layout
   - ForceAtlas2 physics engine through Pyvis.

5. Interactive Features
   - Zooming
   - Panning
   - Hover information
   - Dynamic network exploration

In [59]:
# ============================================================
# TASK 2: NETWORK VISUALISATION
# ============================================================
# ============================================================
# STEP 20: VISUALIZATION INPUT CHECK
# ============================================================

print("\n" + "=" * 70)
print("VISUALIZATION INPUT CHECK")
print("=" * 70)

print(f"G_sub Nodes       : {G_sub.number_of_nodes():,}")
print(f"G_sub Edges       : {G_sub.number_of_edges():,}")

print(f"Centrality Table  : {centrality_df.shape}")

print(f"Top20 Risk Table  : {top20_risk.shape}")

print("\nVisualization inputs verified.")

# ============================================================
# STEP 21: PREPARE VISUALIZATION DATA
# ============================================================

print("\n" + "=" * 80)
print("STEP 21: PREPARE VISUALIZATION DATA")
print("=" * 80)

# Select Top 500 nodes using composite risk score

viz_df = (
    centrality_df
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(500)
    .copy()
)

print(f"\nNodes Selected : {len(viz_df):,}")

print("\nTop 5 Nodes:")

display(
    viz_df[
        [
            "txId",
            "class",
            "risk_score",
            "pagerank"
        ]
    ].head()
)

# ============================================================
# STEP 21A: IDENTIFY TOP 10 RISK NODES
# ============================================================

print("\n" + "=" * 80)
print("STEP 21A: TOP 10 RISK NODES")
print("=" * 80)

top10_risk_nodes = (
    centrality_df
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(10)["txId"]
    .tolist()
)

print("\nTop 10 Risk Nodes:")

for rank, node in enumerate(top10_risk_nodes, start=1):
    print(f"{rank:2d}. {node}")

print("\nTop 10 nodes stored for visualization.")

# ============================================================
# STEP 22: BUILD VISUALIZATION SUBGRAPH
# ============================================================

print("\n" + "=" * 70)
print("STEP 22: BUILD VISUALIZATION SUBGRAPH")
print("=" * 70)

# Extract Top 500 risk nodes

viz_nodes = viz_df["txId"].tolist()

# Create visualization graph

viz_graph = G_sub.subgraph(
    viz_nodes
).copy()

print("\nVisualization Graph Summary:")

print(f"Nodes : {viz_graph.number_of_nodes():,}")
print(f"Edges : {viz_graph.number_of_edges():,}")

print(
    "\nBusiness Interpretation:"
)

print(
    "Visualization focuses on the "
    "highest-risk transactions "
    "while maintaining readability."
)

# ============================================================
# STEP 22A: VERIFY CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("STEP 22A: VISUALIZATION CLASS DISTRIBUTION")
print("=" * 70)

viz_classes = (
    viz_df["class"]
    .value_counts()
)

print(viz_classes)

print("\nPercentage Distribution:")

for cls, count in viz_classes.items():

    pct = count / len(viz_df) * 100

    print(
        f"{cls:<10} : "
        f"{count:>4} "
        f"({pct:.2f}%)"
    )


# ==============================================================================
# CLEAN OLD DASHBOARD FILES
# ==============================================================================

import os

files_to_remove = [
    "fraud_dashboard.html",
    "network_component.html"
]

for file in files_to_remove:

    if os.path.exists(file):

        os.remove(file)

        print(f"Removed: {file}")

print("\nWorkspace cleaned.")


VISUALIZATION INPUT CHECK
G_sub Nodes       : 5,207
G_sub Edges       : 6,431
Centrality Table  : (5207, 13)
Top20 Risk Table  : (20, 13)

Visualization inputs verified.

STEP 21: PREPARE VISUALIZATION DATA

Nodes Selected : 500

Top 5 Nodes:


,txId,class,risk_score,pagerank
3553,38688623,licit,0.619009,0.007192
3924,36364435,licit,0.515375,0.008081
3497,225859042,unknown,0.450000,0.016325
4947,149158766,licit,0.373300,0.014625
2530,232438397,licit,0.359425,0.012412



STEP 21A: TOP 10 RISK NODES

Top 10 Risk Nodes:
 1. 38688623
 2. 36364435
 3. 225859042
 4. 149158766
 5. 232438397
 6. 73227485
 7. 47066538
 8. 70245040
 9. 38538087
10. 38615823

Top 10 nodes stored for visualization.

STEP 22: BUILD VISUALIZATION SUBGRAPH

Visualization Graph Summary:
Nodes : 500
Edges : 958

Business Interpretation:
Visualization focuses on the highest-risk transactions while maintaining readability.

STEP 22A: VISUALIZATION CLASS DISTRIBUTION
class
unknown    278
licit      218
illicit      4
Name: count, dtype: int64

Percentage Distribution:
unknown    :  278 (55.60%)
licit      :  218 (43.60%)
illicit    :    4 (0.80%)
Removed: fraud_dashboard.html
Removed: network_component.html

Workspace cleaned.


 2.5 Network Visualization

The final analytical subgraph contained 5,207 nodes and 6,431 edges.

Direct visualization of the complete network would produce a highly dense graph with limited interpretability.

To improve readability while preserving analytical relevance, the Top 500 transactions ranked by composite risk score were selected for visualization.

**Visualization Network**

- Nodes: 500
- Edges: 958

 **Class Composition**

- Unknown: 278 (55.60%)
- Licit: 218 (43.60%)
- Illicit: 4 (0.80%)

 **Visual Encoding**

- Gold = Top 10 highest-risk transactions
- Red = Illicit transactions
- Green = Licit transactions
- Orange = Unknown transactions
- Node size proportional to PageRank score

An interactive Pyvis network was used to support exploration of transaction relationships, central nodes, and potentially suspicious structures.

TASK 3: FRAUD PATTERN DETECTION

3.1 Objective

The objective of this task is to identify transaction patterns commonly associated with money laundering and financial crime within the analytical subgraph.

Two specific patterns are investigated:

Layering Behaviour

Layering refers to the movement of funds through intermediary transactions in order to obscure the origin of funds. In network terms, layering nodes typically:

- Receive funds from multiple sources
- Redistribute funds to multiple destinations
- Operate across consecutive time steps
- Occupy structurally important network positions

Fan-Out Behaviour

Fan-out behaviour occurs when a transaction distributes funds to many distinct destinations within a short time period.

Such behaviour may indicate:

- Fund dispersion
- Transaction splitting
- Obfuscation strategies
- Early-stage layering activity

Each identified pattern will be cross-referenced against known transaction labels (licit, illicit, and unknown) to determine whether suspicious structural behaviour aligns with known classifications.

In [60]:
# ============================================================
# TASK 3: FRAUD PATTERN DETECTION
# ============================================================

print("TASK 3: FRAUD PATTERN DETECTION")


# STEP 23: DETECT LAYERING BEHAVIOUR

print("\n" + "=" * 70)
print("STEP 24: LAYERING BEHAVIOUR DETECTION")
print("=" * 70)

layering_results = []

for node in G_sub.nodes():

    sources = len(set(G_sub.predecessors(node)))

    destinations = len(set(G_sub.successors(node)))

    node_time = G_sub.nodes[node].get("time_step")

    if sources >= 2 and destinations >= 2:

        neighbouring_times = []

        for pred in G_sub.predecessors(node):

            neighbouring_times.append(
                G_sub.nodes[pred].get("time_step")
            )

        for succ in G_sub.successors(node):

            neighbouring_times.append(
                G_sub.nodes[succ].get("time_step")
            )

        if len(neighbouring_times) > 0:

            time_range = (
                max(neighbouring_times)
                -
                min(neighbouring_times)
            )

            if time_range <= 3:

                layering_results.append({

                    "txId": node,

                    "class": G_sub.nodes[node].get("class"),

                    "sources": sources,

                    "destinations": destinations,

                    "time_steps_span": time_range

                })

layering_df = pd.DataFrame(layering_results)

print(f"\nPotential Layering Nodes Found: {len(layering_df):,}")

display(
    layering_df
    .sort_values(
        ["sources","destinations"],
        ascending=False
    )
    .head(20)
)


# STEP 24A: LAYERING CLASS REVIEW

print("\n" + "=" * 70)
print("STEP 24A: LAYERING CLASS DISTRIBUTION")
print("=" * 70)

print(
    layering_df["class"]
    .value_counts()
)


# STEP 24B: TOP LAYERING CANDIDATES

print("\n" + "=" * 70)
print("STEP 24B: TOP LAYERING CANDIDATES")
print("=" * 70)

layering_df["layering_score"] = (
    layering_df["sources"]
    *
    layering_df["destinations"]
)

top_layering = (
    layering_df
    .sort_values(
        "layering_score",
        ascending=False
    )
    .head(10)
)

display(top_layering)


# STEP 24: FAN-OUT PATTERN DETECTION

print("\n" + "=" * 70)
print("STEP 25: FAN-OUT PATTERN DETECTION")
print("=" * 70)

fanout_results = []

for node in G_sub.nodes():

    destinations = set(
        G_sub.successors(node)
    )

    unique_destinations = len(destinations)

    if unique_destinations >= 5:

        times = [G_sub.nodes[node]["time_step"]]

        for dest in destinations:

            times.append(
                G_sub.nodes[dest]["time_step"]
            )

        fanout_results.append({

            "txId": node,

            "class": G_sub.nodes[node].get("class"),

            "unique_destinations": unique_destinations,

            "time_step_min": min(times),

            "time_step_max": max(times)

        })

fanout_df = pd.DataFrame(fanout_results)

print(
    f"\nPotential Fan-Out Nodes Found: {len(fanout_df):,}"
)

display(
    fanout_df
    .sort_values(
        "unique_destinations",
        ascending=False
    )
    .head(20)
)


# STEP 24A: FAN-OUT CLASS DISTRIBUTION

print("\n" + "=" * 70)
print("STEP 25A: FAN-OUT CLASS DISTRIBUTION")
print("=" * 70)

print(
    fanout_df["class"]
    .value_counts()
)


# STEP 24B: TOP FAN-OUT CANDIDATES

print("\n" + "=" * 70)
print("STEP 25B: TOP FAN-OUT CANDIDATES")
print("=" * 70)

top_fanout = (
    fanout_df
    .sort_values(
        "unique_destinations",
        ascending=False
    )
    .head(10)
)

display(top_fanout)

# STEP 24C: OVERLAP WITH TOP 20 RISK NODES

print("\n" + "=" * 70)
print("STEP 25C: RISK OVERLAP ANALYSIS")
print("=" * 70)

top20_ids = set(top20_risk["txId"])

layering_overlap = (
    set(layering_df["txId"])
    &
    top20_ids
)

fanout_overlap = (
    set(fanout_df["txId"])
    &
    top20_ids
)

print(f"\nLayering Nodes in Top 20 Risk List : {len(layering_overlap)}")

print(sorted(layering_overlap))

print(f"\nFan-Out Nodes in Top 20 Risk List : {len(fanout_overlap)}")

print(sorted(fanout_overlap))

TASK 3: FRAUD PATTERN DETECTION

STEP 24: LAYERING BEHAVIOUR DETECTION

Potential Layering Nodes Found: 63


,txId,class,sources,destinations,time_steps_span
47,38688623,licit,98,15,0
51,36364435,licit,97,11,0
31,38615823,licit,94,2,0
49,73227485,licit,94,2,0
22,47066538,licit,93,4,0
7,38538087,licit,80,5,0
52,98034161,licit,79,2,0
18,35202200,licit,77,5,0
50,73031052,licit,48,2,0
21,200649629,licit,45,2,0



STEP 24A: LAYERING CLASS DISTRIBUTION
class
licit      40
unknown    23
Name: count, dtype: int64

STEP 24B: TOP LAYERING CANDIDATES


,txId,class,sources,destinations,time_steps_span,layering_score
47,38688623,licit,98,15,0,1470
51,36364435,licit,97,11,0,1067
7,38538087,licit,80,5,0,400
18,35202200,licit,77,5,0,385
22,47066538,licit,93,4,0,372
49,73227485,licit,94,2,0,188
31,38615823,licit,94,2,0,188
52,98034161,licit,79,2,0,158
50,73031052,licit,48,2,0,96
21,200649629,licit,45,2,0,90



STEP 25: FAN-OUT PATTERN DETECTION

Potential Fan-Out Nodes Found: 160


,txId,class,unique_destinations,time_step_min,time_step_max
29,102570,unknown,122,36,36
102,2347070,unknown,32,36,36
83,10795931,unknown,27,42,42
21,7278020,unknown,27,42,42
22,101617193,unknown,26,5,5
52,14360522,unknown,26,42,42
155,42728266,unknown,24,5,5
37,2070455,licit,23,36,36
33,9605879,unknown,23,42,42
53,73958,unknown,23,42,42



STEP 25A: FAN-OUT CLASS DISTRIBUTION
class
unknown    95
licit      65
Name: count, dtype: int64

STEP 25B: TOP FAN-OUT CANDIDATES


,txId,class,unique_destinations,time_step_min,time_step_max
29,102570,unknown,122,36,36
102,2347070,unknown,32,36,36
83,10795931,unknown,27,42,42
21,7278020,unknown,27,42,42
22,101617193,unknown,26,5,5
52,14360522,unknown,26,42,42
155,42728266,unknown,24,5,5
37,2070455,licit,23,36,36
33,9605879,unknown,23,42,42
53,73958,unknown,23,42,42



STEP 25C: RISK OVERLAP ANALYSIS

Layering Nodes in Top 20 Risk List : 9
[757859, 35202200, 36364435, 38538087, 38615823, 38688623, 47066538, 73227485, 98034161]

Fan-Out Nodes in Top 20 Risk List : 8
[757859, 9605879, 10003128, 28126756, 35202200, 36364435, 38538087, 38688623]



Task 3 Findings Section

3.2 Layering Behaviour Detection

Layering analysis identified 63 transactions that received funds from multiple sources and redistributed them onward to multiple destinations.

The strongest layering candidates included:

| Transaction ID | Sources | Destinations | Class |
| -------------- | ------- | ------------ | ----- |
| 38688623       | 98      | 15           | Licit |
| 36364435       | 97      | 11           | Licit |
| 38538087       | 80      | 5            | Licit |
| 35202200       | 77      | 5            | Licit |
| 47066538       | 93      | 4            | Licit |

Class label distribution of identified layering nodes:

* Licit: 40
* Unknown: 23
* Illicit: 0

These transactions exhibit aggregation and redistribution behaviour consistent with potential layering activity. Although none of the detected layering nodes were labelled illicit, their structural characteristics suggest they occupy influential positions within the transaction network and warrant further investigation.

3.3 Fan-Out Pattern Detection

Fan-out analysis identified 160 transactions that distributed funds to multiple distinct destinations.

The strongest fan-out candidates included:

| Transaction ID | Unique Destinations | Class   |
| -------------- | ------------------- | ------- |
| 102570         | 122                 | Unknown |
| 2347070        | 32                  | Unknown |
| 10795931       | 27                  | Unknown |
| 7278020        | 27                  | Unknown |
| 101617193      | 26                  | Unknown |

Class label distribution of identified fan-out nodes:

* Unknown: 95
* Licit: 65
* Illicit: 0

From a compliance perspective, these transactions demonstrate unusually high fund-dispersion behaviour that may be associated with transaction splitting, obfuscation, or early-stage layering activity.

3.4 Cross-Reference with Risk Scoring

To validate the analytical framework, identified fraud patterns were compared against the Top 20 highest-risk transactions.

Results showed:

* 9 layering nodes appeared within the Top 20 risk-ranked transactions.
* 8 fan-out nodes appeared within the Top 20 risk-ranked transactions.

Most notably, the following transactions were identified simultaneously as:

* High-risk transactions
* Layering candidates
* Fan-out candidates

| Transaction ID |
| -------------- |
| 35202200       |
| 36364435       |
| 38538087       |
| 38688623       |

These transactions represent the strongest candidates for enhanced compliance review because they exhibit multiple independent indicators of structurally suspicious behaviour.

3.5 Key Fraud Detection Insight

The majority of suspicious network structures were associated with transactions labelled as licit or unknown rather than illicit.

This finding demonstrates that network topology can reveal potential risk signals that are not captured by transaction labels alone. Consequently, structural network analysis provides valuable supplementary intelligence for anti-money laundering investigations and transaction monitoring programs.


TASK 4: BUSINESS REPORT

Audience: Head of Compliance

Objective:
Summarize key fraud risks identified through network analysis,
highlight priority transactions for review,
provide actionable recommendations,
and document analytical limitations.

Business Report to the Head of Compliance

Subject: Findings from Bitcoin Transaction Network Analysis

This analysis examined a network of 5,207 Bitcoin transactions and 6,431 payment flows to identify transaction patterns that may indicate elevated compliance risk. The review focused on network behaviour rather than transaction values alone, allowing the identification of transactions that occupy influential positions within the payment network.

Three transactions emerged as the highest-risk nodes in the network. Transaction **38688623** recorded the highest composite risk score (0.619) and was connected to 98 incoming transactions and 15 outgoing transactions. Transaction **36364435** recorded the second-highest risk score (0.515), receiving funds from 97 sources and distributing them to 11 destinations. Transaction **225859042** recorded the third-highest risk score (0.450) and occupied a highly influential position within the network despite being classified as an unknown transaction. These transactions warrant immediate review because they play central roles in the movement of funds across the network.

The most suspicious cluster identified consisted of transactions **35202200**, **36364435**, **38538087**, and **38688623**. These transactions were simultaneously identified as high-risk nodes, layering candidates, and fan-out candidates. In practical terms, they received funds from many sources and redistributed funds onward to multiple destinations. Such behaviour can be consistent with attempts to obscure transaction origins and complicate transaction tracing. While these transactions are currently labelled as licit, their structural behaviour suggests that additional scrutiny is appropriate.

The analysis also identified a large number of unknown transactions exhibiting fan-out behaviour. The strongest example was transaction **102570**, which distributed funds to 122 distinct destinations within the same time period. Although unlabelled, this pattern is unusual and may indicate fund dispersion activity that merits further investigation.

Recommendations

Recommendation 1: Conduct enhanced monitoring and manual review of the identified high-risk transactions, particularly transactions 38688623, 36364435, and 225859042.

Owner:Transaction Monitoring Team Lead.

Recommendation 2:Prioritize investigation of unknown transactions exhibiting large fan-out patterns and assess whether existing monitoring rules adequately capture this behaviour.

Owner: Financial Crime Investigation Team Lead.

Data Limitation

A key limitation of this analysis is that a substantial proportion of transactions within the analytical subgraph were classified as unknown rather than licit or illicit. This limits the ability to directly validate whether suspicious network structures correspond to confirmed financial crime and may result in potentially risky transactions remaining unidentified by label-based approaches alone.

Overall, the findings demonstrate that network analysis provides valuable intelligence beyond transaction classifications and can help identify potentially suspicious activity requiring enhanced compliance review.


In [63]:
# ============================================================
# TASK 5: PYTHON DASHBOARD
# STEP 26: DASHBOARD DATA PREPARATION
# ============================================================
print("centrality_df:", centrality_df.shape)
print("top20_risk:", top20_risk.shape)
print("viz_graph:", viz_graph.number_of_nodes(), viz_graph.number_of_edges())

!pip install pyvis plotly -q
print("pyvis plotly installation successful.")

from pyvis.network import Network
import plotly.express as px
import plotly.io as pio
import pandas as pd
import shutil
import os

print("Libraries loaded successfully.")

# KPI VALUES

total_nodes = G_sub.number_of_nodes()
total_edges = G_sub.number_of_edges()

illicit_nodes = (
    centrality_df["class"] == "illicit"
).sum()

licit_nodes = (
    centrality_df["class"] == "licit"
).sum()

unknown_nodes = (
    centrality_df["class"] == "unknown"
).sum()

# Top 20 Risk Table

dashboard_table = (
    top20_risk[
        [
            "txId",
            "class",
            "risk_score",
            "pagerank",
            "betweenness_centrality"
        ]
    ]
    .copy()
)

dashboard_table.columns = [
    "Transaction ID",
    "Class",
    "Risk Score",
    "PageRank",
    "Betweenness Centrality"
]

risk_table_html = (
    dashboard_table
    .round(6)
    .to_html(
        index=False,
        classes="risk-table"
    )
)
print("Dashboard data prepared.")

# ============================================================
# STEP 27: BUILD EXECUTIVE NETWORK GRAPH
# ============================================================

net = Network(
    height="850px",
    width="100%",
    bgcolor="#0B132B",
    font_color="white",
    directed=True
)

node_lookup = centrality_df.set_index("txId")

top10_nodes = (
    centrality_df
    .nlargest(10, "risk_score")
    ["txId"]
    .tolist()
)

TOP10_COLOR = "#f4c542"
ILLICIT_COLOR = "#e63946"
LICIT_COLOR = "#2e7d32"
UNKNOWN_COLOR = "#85aec4"

for node in viz_graph.nodes():

    node_class = viz_graph.nodes[node]["class"]
    time_step = viz_graph.nodes[node]["time_step"]

    risk_score = node_lookup.loc[node, "risk_score"]
    pagerank = node_lookup.loc[node, "pagerank"]
    betweenness = node_lookup.loc[node, "betweenness_centrality"]

    if node in top10_nodes:
        color = TOP10_COLOR
        size = 45

    elif node_class == "illicit":
        color = ILLICIT_COLOR
        size = 18 + (risk_score * 40)

    elif node_class == "licit":
        color = LICIT_COLOR
        size = 12 + (risk_score * 30)

    else:
        color = UNKNOWN_COLOR
        size = 10 + (risk_score * 25)

    tooltip = f"""
    <b>Transaction ID:</b> {node}<br>
    <b>Class:</b> {node_class}<br>
    <b>Time Step:</b> {time_step}<br>
    <b>Risk Score:</b> {risk_score:.4f}<br>
    <b>PageRank:</b> {pagerank:.6f}<br>
    <b>Betweenness:</b> {betweenness:.6f}
    """

    net.add_node(
        str(node),
        title=tooltip,
        color=color,
        size=size,
        label=""
    )

for source, target in viz_graph.edges():

    net.add_edge(
        str(source),
        str(target),
        color="#5A6A85"
    )

net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -120,
      "springLength": 180
    },
    "solver": "forceAtlas2Based"
  }
}
""")

net.save_graph("network_component.html")

print("Network graph built.")

# ==============================================================================
# STEP 28: BUILD TIME ACTIVITY DATA
# ==============================================================================

time_activity = (
    centrality_df
    .groupby(["time_step", "class"])
    .size()
    .reset_index(name="transactions")
)

fig = px.line(
    time_activity,
    x="time_step",
    y="transactions",
    color="class",
    markers=True,
    title="Transaction Activity Across Time Steps"
)

fig.update_layout(
    template="plotly_white",
    height=500
)

time_chart_html = pio.to_html(
    fig,
    include_plotlyjs="cdn",
    full_html=False
)

print("Time chart built.")


# ============================================================
# STEP 29: BUILD  DASHBOARD
# ============================================================

print("=" * 70)
print("BUILDING  DASHBOARD")
print("=" * 70)

# ============================================================
# NETWORK GRAPH EMBED
# ============================================================

with open(
    "network_component.html",
    "r",
    encoding="utf-8"
) as f:

    network_html = f.read()
dashboard_html = f"""

<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>
Elliptic Bitcoin Fraud Dashboard
</title>

<style>

body {{
background:#d6e8f5;
font-family:Arial,sans-serif;
margin:0;
}}

.header {{
background:#0d1b2a;
color:white;
padding:25px;
}}

.author {{
float:right;
color:#4FC3F7;
font-weight:bold;
}}

.kpis {{
    display:flex;
    gap:20px;
    margin:25px;
    flex-wrap:wrap;
}}

.card {{
    flex:1;
    min-width:180px;

    background:white;

    border-radius:12px;

    padding:25px;

    text-align:center;

    box-shadow:0 4px 12px rgba(0,0,0,0.15);

    border-top:5px solid #4FC3F7;
}}

.card h2 {{
    margin:0;

    font-size:36px;

    color:#0d1b2a;

    font-weight:bold;
}}

.card p {{
    margin-top:10px;

    font-size:15px;

    color:#555;
}}

.section {{
background:white;
margin:20px;
padding:20px;
border-radius:10px;
box-shadow:0 2px 8px rgba(0,0,0,0.1);
}}


.risk-table {{
    width:100%;
    border-collapse:collapse;
}}

.risk-table th {{
    background:#1b2a3b;
    color:white;
    padding:12px;
}}

.risk-table td {{
    padding:10px;
    border-bottom:1px solid #dddddd;
}}

.risk-table tr:hover {{
    background:#f2f7fb;
}}

table {{
width:100%;
border-collapse:collapse;
}}

th {{
background:#1b2a3b;
color:white;
padding:10px;
}}

td {{
padding:10px;
border-bottom:1px solid #ccc;
}}


</style>

</head>

<body>

<div class="header">

<div class="author">
Ajibola Abiodun Iremide
</div>

<h1>
ELLIPTIC BITCOIN FRAUD NETWORK ANALYSIS
</h1>

<h3>
Who Are The Riskiest Nodes And How Are They Connected?
</h3>

</div>

<div class="kpis">

<div class="card">
<h2>{total_nodes:,}</h2>
<p>Total Nodes</p>
</div>

<div class="card">
<h2>{total_edges:,}</h2>
<p>Total Edges</p>
</div>

<div class="card">
<h2>{illicit_nodes:,}</h2>
<p>Illicit Nodes</p>
</div>

<div class="card">
<h2>{licit_nodes:,}</h2>
<p>Licit Nodes</p>
</div>

<div class="card">
<h2>{unknown_nodes:,}</h2>
<p>Unknown Nodes</p>
</div>

</div>

<div class="section">

<h2>Executive Findings</h2>

<ul>
<li>Highest Risk Node: 38688623 (Risk Score = 0.619)</li>
<li>Strongest Fan-Out Node: 102570 (122 destinations)</li>
<li>Strongest Layering Node: 38688623 (98 sources → 15 destinations)</li>
<li>9 layering nodes overlap with Top 20 risk transactions</li>
<li>8 fan-out nodes overlap with Top 20 risk transactions</li>
</ul>

</div>

<div class="section">

<h2>Interactive Network Graph</h2>

{network_html}

</div>

<div class="section">

<h2>Top 20 Risk Transactions</h2>

{risk_table_html}

</div>

<div class="section">

<h2>Transaction Activity Across Time Steps</h2>

{time_chart_html}

</div>

<div class="section">

<h2>Network Legend</h2>

 Gold = Top 10 Risk Nodes<br>
 Red = Illicit<br>
 Green = Licit<br>
 Blue-Grey = Unknown

</div>

</body>
</html>

"""

with open(
    "fraud_dashboard.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(dashboard_html)

print("Dashboard created successfully.")

centrality_df: (5207, 13)
top20_risk: (20, 13)
viz_graph: 500 958
pyvis plotly installation successful.
Libraries loaded successfully.
Dashboard data prepared.
Network graph built.
Time chart built.
BUILDING  DASHBOARD
Dashboard created successfully.


In [64]:

# ============================================================
# STEP 28: SAVE & DOWNLOAD
# ============================================================
from google.colab import files

files.download("fraud_dashboard.html")

shutil.copy(
    "fraud_dashboard.html",
    "/content/drive/MyDrive/HNG14_Stage7_Abiodun_Ajibola/fraud_dashboard.html"
)

print("Dashboard saved to Google Drive.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dashboard saved to Google Drive.
